# AWS Transcribe x Finetunado em relação ao ground truth

In [ ]:
import os, json, subprocess, warnings, re, shutil, tempfile, sys
from pathlib import Path
from tqdm import tqdm
import torch
from pyannote.audio import Pipeline, Model
from itertools import groupby

# reduzir ruído de warnings
warnings.filterwarnings("ignore")

# === compat: dscore antigo + numpy novo ===
import numpy as np
if not hasattr(np, "int"):   np.int = int
if not hasattr(np, "float"): np.float = float
if not hasattr(np, "bool"):  np.bool = bool

# === dscore (scorelib) ===
from dscore_master.scorelib.score import score
from dscore_master.scorelib.rttm import load_rttm
from dscore_master.scorelib.uem import load_uem

# =========================
# CONFIG (ajuste aqui)
# =========================
# raiz da estrutura que vc mostrou
# Pasta raiz com os RTTMs gerados pelo pipeline de comparação
# Estrutura esperada: results_dir/{audios,rttms/TRUTH,rttms/FINETUNADO,rttms/DIARIZATION,rttms/AWSTRANSCRIBE}
ROOT = Path(os.environ.get("RESULTS_DIR", "results"))

# pastas
AUD_DIR  = ROOT / "audios"
REF_DIR  = ROOT / "rttms" / "TRUTH"          # (GROUND-TRUTH)  <<< usado por dscore_fold
UEM_DIR  = ROOT / "uems"                     #                 <<< usado por dscore_fold
LIST_TXT = ROOT / "uris.txt"                 # lista de URIs (sem .wav)

# sistemas:
BASE_DIR = ROOT / "rttms" / "AWSTRANSCRIBE"  # já existe, será comparado com TRUTH
FT_DIR   = ROOT / "rttms" / "FINETUNADO"     # vamos gerar agora com o FT
OLD_DIR  = ROOT / "rttms" / "DIARIZATION"

# onde salvar agregados/relatórios
SCORE_DIR = ROOT / "results"                 # <<< usado por dscore_fold
SCORE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def read_uris(p: Path):
    return [ln.strip() for ln in p.read_text().splitlines() if ln.strip()]

def concat(files, out_path: Path):
    with out_path.open("w") as w:
        for f in files:
            if Path(f).exists():
                for line in Path(f).read_text().splitlines():
                    if line.strip():  # ignora linhas vazias
                        w.write(line + "\n")

def run(cmd, capture=False, shell=False):
    try:
        return subprocess.run(
            " ".join(cmd) if shell else cmd,
            check=True, text=True, capture_output=capture, shell=shell
        )
    except subprocess.CalledProcessError as e:
        print("\n[CMD ERRO]:", e.args if hasattr(e, "args") else cmd)
        if e.stdout: print("\n[STDOUT]\n", e.stdout)
        if e.stderr: print("\n[STDERR]\n", e.stderr)
        raise

def diarize_many(pipe, uris, out_dir: Path):
    from pyannote.core import Annotation
    out_dir.mkdir(parents=True, exist_ok=True)
    for uri in tqdm(uris, desc=f"diarize->{out_dir.name}"):
        wav = AUD_DIR / f"{uri}.wav"
        if not wav.exists():
            raise FileNotFoundError(wav)
        diar = pipe(str(wav), num_speakers=2)
        diar.uri = uri  # seta o identificador
        with (out_dir / f"{uri}.rttm").open("w") as f:
            diar.write_rttm(f)

def _get_der(global_scores):
    # Scores pode ser uma namedtuple/obj com atributo 'der' ou índice fixo
    if hasattr(global_scores, "der"):
        return float(global_scores.der)
    if hasattr(global_scores, "DER"):
        return float(global_scores.DER)
    try:
        # fallback: posição 1 é o DER em várias versões
        return float(global_scores[1])
    except Exception:
        raise RuntimeError(f"Não consegui extrair DER de {type(global_scores)}: {global_scores}")

def load_turns_flat(rttm_path: Path):
    turns, _speaker_ids, _file_ids = load_rttm(str(rttm_path))  # <<-- UNPACK AQUI
    # 'turns' já deve ser list[Turn]; se por algum motivo vier aninhado, achata:
    if isinstance(turns, list) and turns and isinstance(turns[0], list):
        turns = [t for sub in turns for t in sub]
    # ordena (opcional) pra estabilidade
    try:
        turns.sort(key=lambda t: (t.file_id, t.tbeg))
    except Exception:
        pass
    return turns

def dscore_fold_2(sys_dir: Path, tag: str):
    sys_all = SCORE_DIR / f"{tag}_sys.rttm"
    ref_all = SCORE_DIR / f"{tag}_ref.rttm"
    uem_all = SCORE_DIR / f"{tag}_ref.uem"

    concat(sorted(sys_dir.glob("*.rttm")), sys_all)
    concat(sorted(REF_DIR.glob("*.rttm")), ref_all)
    concat(sorted(UEM_DIR.glob("*.uem")),  uem_all)

    ref_turns = load_turns_flat(ref_all)
    sys_turns = load_turns_flat(sys_all)
    uem_obj   = load_uem(str(uem_all))

    out = {}
    for name, collar, ignore_ov in [("forgiving",0.25,True), ("fair",0.25,False), ("full",0.0,False)]:
        file_scores, global_scores = score(
            ref_turns=ref_turns,
            sys_turns=sys_turns,
            uem=uem_obj,
            collar=collar,
            ignore_overlaps=ignore_ov
        )
        der = _get_der(global_scores)
        out[name] = der
        (SCORE_DIR / f"{tag}_{name}.txt").write_text(
            f"DER={der:.4f}  collar={collar}  ignore_overlap={ignore_ov}\n"
        )
    return out

In [ ]:
from diarizers import SegmentationModel

# 1) ler URIs
uris = read_uris(LIST_TXT)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

ft_pipe = Pipeline.from_pretrained("fatymatariq/speaker-diarization-3.1").to(device)

seg_model = SegmentationModel().from_pretrained("models/segmentation_finetuned",)
seg_model = seg_model.to_pyannote_model()

ft_pipe._segmentation.model = seg_model.to(device)

FT_DIR.mkdir(parents=True, exist_ok=True)
diarize_many(ft_pipe, uris, FT_DIR)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

diar_pipe = Pipeline.from_pretrained("fatymatariq/speaker-diarization-3.1").to(device)

OLD_DIR.mkdir(parents=True, exist_ok=True)
diarize_many(diar_pipe, uris, OLD_DIR)

In [ ]:
aws_scores = dscore_fold_2(BASE_DIR, "aws")   # usa REF_DIR (TRUTH) e UEM_DIR globais
ft_scores  = dscore_fold_2(FT_DIR,   "ft")
diar_scores= dscore_fold_2(OLD_DIR, "diar")

print("\nDER global (%):")
print("AWSTRANSCRIBE:", aws_scores.get('full'))
print("DIARIZATION  :", diar_scores.get('full'))
print("FINETUNADO   :", ft_scores.get('full'))

In [ ]:
# %% [markdown]
# Compara 3 sistemas (FINETUNADO, DIARIZATION, AWSTRANSCRIBE) contra o TRUTH
# por arquivo (URI) usando um DER_like (MISS + FA + CONF / tempo_ref).
# Saídas:
#   - rttm_similarity_report.csv  -> linhas por URI x sistema com métricas + best_system
#   - rttm_similarity_summary.csv -> contagem e % de vitórias por sistema

# %%
from pathlib import Path
import re
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

# -----------------------------
# Configuração das pastas
# -----------------------------
DIR_TRUTH        = ROOT / "rttms" / "TRUTH"
DIR_FINETUNADO   = ROOT / "rttms" / "FINETUNADO"
DIR_DIARIZATION  = ROOT / "rttms" / "DIARIZATION"
DIR_AWST         = ROOT / "rttms" / "AWSTRANSCRIBE"

OUTPUT_DIR = Path(".")
FRAME_STEP = 0.01  # 10 ms

# -----------------------------
# Parser simples de RTTM
# -----------------------------
RTTM_LINE = re.compile(
    r"^SPEAKER\s+(?P<uri>\S+)\s+\S+\s+(?P<start>\d+(?:\.\d+)?)\s+(?P<dur>\d+(?:\.\d+)?)\s+\S+\s+\S+\s+(?P<spk>\S+)"
)

def parse_rttm(path: Path) -> Dict[str, List[Tuple[float, float, str]]]:
    """
    Lê 1 RTTM -> dict com ÚNICA key (o próprio nome base do arquivo, sem extensão) mapeando
    para [(start, end, speaker), ...].
    Mantemos uma assinatura uri=nome_do_arquivo_sem_extensão para ser consistente.
    """
    data: Dict[str, List[Tuple[float, float, str]]] = {}
    if not path.exists():
        return data
    uri_from_name = path.stem
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            m = RTTM_LINE.match(line)
            if not m:
                continue
            start = float(m.group("start"))
            dur   = float(m.group("dur"))
            end   = start + dur
            spk   = m.group("spk")
            data.setdefault(uri_from_name, []).append((start, end, spk))
    for uri in data:
        data[uri].sort(key=lambda x: x[0])
    return data

# -----------------------------
# Alinhamento em frames
# -----------------------------
def align_timelines(ref_segs, sys_segs, step: float = 0.01):
    """
    Cria arrays alinhados ref_idx e sys_idx (mesmo grid temporal).
    Cada posição é índice de speaker (>=0) ou -1 (silêncio).
    Retorna: ref_idx, sys_idx, ref_labels, sys_labels, t0, step
    """
    times = []
    for s, e, _ in ref_segs:
        times.extend([s, e])
    for s, e, _ in sys_segs:
        times.extend([s, e])

    if not times:
        # Sem segmentos em ambos -> arrays vazios
        return (np.empty((0,), dtype=np.int32),
                np.empty((0,), dtype=np.int32),
                [], [], 0.0, step)

    t0 = min(times)
    t1 = max(times)
    n_frames = int(np.ceil((t1 - t0) / step))
    ref = np.full((n_frames,), -1, dtype=np.int32)
    sys = np.full((n_frames,), -1, dtype=np.int32)

    ref_spk_to_idx: Dict[str, int] = {}
    sys_spk_to_idx: Dict[str, int] = {}
    ref_labels: List[str] = []
    sys_labels: List[str] = []

    for s, e, spk in ref_segs:
        if spk not in ref_spk_to_idx:
            ref_spk_to_idx[spk] = len(ref_labels)
            ref_labels.append(spk)
        i0 = max(0, int(np.floor((s - t0) / step)))
        i1 = min(n_frames, int(np.ceil((e - t0) / step)))
        if i1 > i0:
            ref[i0:i1] = ref_spk_to_idx[spk]

    for s, e, spk in sys_segs:
        if spk not in sys_spk_to_idx:
            sys_spk_to_idx[spk] = len(sys_labels)
            sys_labels.append(spk)
        i0 = max(0, int(np.floor((s - t0) / step)))
        i1 = min(n_frames, int(np.ceil((e - t0) / step)))
        if i1 > i0:
            sys[i0:i1] = sys_spk_to_idx[spk]

    return ref, sys, ref_labels, sys_labels, t0, step

# -----------------------------
# DER_like (MISS + FA + CONF / total_ref)
# -----------------------------
def compute_der_like(ref_idx: np.ndarray,
                     sys_idx: np.ndarray,
                     ref_labels: List[str],
                     sys_labels: List[str],
                     step: float) -> Dict[str, float]:
    if ref_idx.size == 0 and sys_idx.size == 0:
        return {"MISS": 0.0, "FA": 0.0, "CONF": 0.0, "TOTAL_REF": 0.0, "DER": 0.0}

    assert ref_idx.shape == sys_idx.shape, "Arrays precisam ter o mesmo tamanho"
    frame_dur = step

    ref_speech = ref_idx >= 0
    sys_speech = sys_idx >= 0
    both = ref_speech & sys_speech

    total_ref_time = ref_speech.sum() * frame_dur
    miss_time = (ref_speech & (~sys_speech)).sum() * frame_dur
    fa_time   = ((~ref_speech) & sys_speech).sum() * frame_dur

    # Se não há sobreposição com fala nos dois, conf = 0
    if not both.any() or len(ref_labels) == 0 or len(sys_labels) == 0:
        conf_time = 0.0
        der = (miss_time + fa_time + conf_time) / total_ref_time if total_ref_time > 0 else float("inf")
        return {"MISS": miss_time, "FA": fa_time, "CONF": conf_time, "TOTAL_REF": total_ref_time, "DER": der}

    # Matriz de sobreposição (frames) entre speakers ref x sys nas regiões "both"
    R = len(ref_labels)
    S = len(sys_labels)
    overlap = np.zeros((R, S), dtype=np.float64)
    ref_both = ref_idx[both]
    sys_both = sys_idx[both]
    for r in range(R):
        r_mask = ref_both == r
        if not r_mask.any():
            continue
        for s in range(S):
            overlap[r, s] += (r_mask & (sys_both == s)).sum()

    if overlap.size == 0:
        conf_time = 0.0
        der = (miss_time + fa_time + conf_time) / total_ref_time if total_ref_time > 0 else float("inf")
        return {"MISS": miss_time, "FA": fa_time, "CONF": conf_time, "TOTAL_REF": total_ref_time, "DER": der}

    # Hungarian para maximizar o acerto (minimizando -overlap)
    R2 = max(R, S)
    cost = np.zeros((R2, R2), dtype=np.float64)
    cost[:R, :S] = -overlap
    row_ind, col_ind = linear_sum_assignment(cost)
    correct_overlap_frames = 0
    for r, c in zip(row_ind, col_ind):
        if r < R and c < S:
            correct_overlap_frames += int(overlap[r, c])

    both_time   = both.sum() * frame_dur
    correct_time = correct_overlap_frames * frame_dur
    conf_time    = both_time - correct_time

    der = (miss_time + fa_time + conf_time) / total_ref_time if total_ref_time > 0 else float("inf")
    return {"MISS": miss_time, "FA": fa_time, "CONF": conf_time, "TOTAL_REF": total_ref_time, "DER": der}

# -----------------------------
# Loop principal
# -----------------------------
def calc_for_all():
    truth_files = sorted(DIR_TRUTH.glob("*.rttm"))
    if not truth_files:
        raise FileNotFoundError(f"Nenhum .rttm encontrado em {DIR_TRUTH.resolve()}")

    systems = {
        "FINETUNADO":   DIR_FINETUNADO,
        "DIARIZATION":  DIR_DIARIZATION,
        "AWSTRANSCRIBE":DIR_AWST,
    }

    rows = []
    for truth_rttm in truth_files:
        uri = truth_rttm.stem

        # Carrega REF
        ref_map = parse_rttm(truth_rttm)
        ref_segs = ref_map.get(uri, [])

        # Para cada sistema, tenta carregar o URI correspondente
        for sys_name, sys_dir in systems.items():
            sys_path = (sys_dir / f"{uri}.rttm")
            if sys_path.exists():
                sys_map = parse_rttm(sys_path)
                sys_segs = sys_map.get(uri, [])
                ref_idx, sys_idx, ref_labels, sys_labels, _, _ = align_timelines(ref_segs, sys_segs, step=FRAME_STEP)
                metrics = compute_der_like(ref_idx, sys_idx, ref_labels, sys_labels, step=FRAME_STEP)
                der_val = metrics["DER"]
            else:
                # Se não existe, marca DER como infinito (não pode ganhar)
                metrics = {"MISS": np.nan, "FA": np.nan, "CONF": np.nan, "TOTAL_REF": np.nan, "DER": float("inf")}
                der_val = metrics["DER"]

            rows.append({
                "uri": uri,
                "system": sys_name,
                "DER_like": None if np.isinf(der_val) else round(der_val, 6),
                "MISS_sec": None if np.isnan(metrics["MISS"]) else round(metrics["MISS"], 3),
                "FA_sec":   None if np.isnan(metrics["FA"])   else round(metrics["FA"], 3),
                "CONF_sec": None if np.isnan(metrics["CONF"]) else round(metrics["CONF"], 3),
                "TOTAL_REF_sec": None if np.isnan(metrics["TOTAL_REF"]) else round(metrics["TOTAL_REF"], 3),
                "missing_file": (not (sys_dir / f"{uri}.rttm").exists())
            })

    df = pd.DataFrame(rows)

    # Melhor por URI: escolher o menor DER_like (ignorando NaN; +inf já virou None)
    # Para garantir desempate estável, ordena por DER_like e depois por nome de sistema
    tmp = df.copy()
    # Substitui None por +inf só para a ordenação/argmin
    tmp["DER_sort"] = tmp["DER_like"].apply(lambda x: float("inf") if x is None else x)

    best_df = (
        tmp.sort_values(["uri", "DER_sort", "system"])
           .groupby("uri", as_index=False)
           .first()[["uri", "system", "DER_like"]]
           .rename(columns={"system": "best_system", "DER_like": "best_DER_like"})
    )

    report = df.merge(best_df, on="uri", how="left")

    # Resumo: contagem e % por sistema
    total_uris = best_df.shape[0]
    summary = (
        best_df["best_system"]
        .value_counts()
        .rename_axis("system")
        .reset_index(name="count")
        .sort_values("system")
    )
    summary["percent"] = (summary["count"] / total_uris * 100.0).round(2)

    # Salva CSVs
    out_report  = OUTPUT_DIR / "rttm_similarity_report.csv"
    out_summary = OUTPUT_DIR / "rttm_similarity_summary.csv"
    report.to_csv(out_report, index=False)
    summary.to_csv(out_summary, index=False)

    # Prints
    print(f"Total de arquivos (URIs) avaliados: {total_uris}\n")
    # Mostra vencedor por URI (primeiras 30 linhas para não poluir demais)
    print("Vencedor por URI (amostra):")
    print(best_df.head(30).to_string(index=False))
    print("\nResumo de vitórias por sistema:")
    print(summary.to_string(index=False))
    print(f"\nCSV detalhado salvo em: {out_report.resolve()}")
    print(f"CSV resumo salvo em:    {out_summary.resolve()}")

    return report, summary, best_df

# %%
# Executar
report_df, summary_df, winners_df = calc_for_all()


# Testes

In [ ]:
from pydub import AudioSegment
from typing import List, Dict
import numpy as np

### functions

In [ ]:
def audiosegment_to_ndarray(audio_segment, target_sample_rate=16000):
    """
    Converte um pydub.AudioSegment para numpy.ndarray compatível com Whisper.
    """
    audio = (
        audio_segment.set_frame_rate(target_sample_rate)
        .set_channels(1)
        .set_sample_width(2)
    )
    samples = np.array(audio.get_array_of_samples()).astype(np.float32) / 32768.0
    return samples

def segmenta_por_speaker(
    file_path: str, diarization, min_duration: float = 1.0
) -> List[Dict]:
    """
    Corta o áudio em segmentos por speaker, agrupando falas consecutivas do mesmo speaker
    e retornando apenas os segmentos com mais de `min_duration` segundos,
    já convertidos para numpy.ndarray.

    Parâmetros:
    - file_path (str): Caminho do arquivo de áudio .wav
    - diarization (pyannote.core.Annotation): Resultado da diarização
    - min_duration (float): Duração mínima (em segundos) do segmento para ser considerado

    Retorna:
    - List[Dict]: Lista com 'name', 'start', 'end', 'audio' (como np.ndarray)
    """
    audio = AudioSegment.from_wav(file_path)

    segments = []
    current_speaker = None
    current_start = None
    current_end = None

    for turn in diarization.itertracks(yield_label=True):
        start, end, speaker = turn[0].start, turn[0].end, turn[2]

        if speaker == current_speaker:
            current_end = end
        else:
            if current_speaker is not None:
                duration = current_end - current_start
                if duration >= min_duration:
                    segment_audio = audio[
                        int(current_start * 1000) : int(current_end * 1000)
                    ]
                    segments.append(
                        {
                            "speaker": current_speaker,
                            "startTime": current_start,
                            "endTime": current_end,
                            "audio": audiosegment_to_ndarray(segment_audio),
                        }
                    )
            current_speaker = speaker
            current_start = start
            current_end = end

    # Adiciona o último segmento
    if current_speaker is not None:
        duration = current_end - current_start
        if duration >= min_duration:
            segment_audio = audio[int(current_start * 1000) : int(current_end * 1000)]
            segments.append(
                {
                    "speaker": current_speaker,
                    "startTime": current_start,
                    "endTime": current_end,
                    "audio": audiosegment_to_ndarray(segment_audio),
                }
            )

    return segments

In [ ]:
from pyannote.core import Annotation, Segment

def segments_to_annotation(segments, uri="unknown"):
    """
    segments: List[Dict] com chaves {speaker, startTime, endTime}
    """
    ann = Annotation(uri=uri)
    for s in segments:
        start = float(s["startTime"]); end = float(s["endTime"])
        if end > start:
            ann[Segment(start, end)] = str(s["speaker"])
    return ann

from pathlib import Path

def save_rttm(annotation, out_path: str | Path):
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        annotation.write_rttm(f)

def diarize_to_annotation(pipeline, audio_path, num_speakers=2, min_duration=1.0):
    diarization = pipeline(audio_path, num_speakers=num_speakers)
    segs = segmenta_por_speaker(audio_path, diarization, min_duration=min_duration)
    ann = segments_to_annotation(segs, uri=Path(audio_path).stem)
    return ann

def run_model_over_list(pipeline, audio_list, num_speakers=2, min_duration=1.0):
    preds = {}  # uri -> Annotation
    for wav in audio_list:
        try:
            ann = diarize_to_annotation(pipeline, wav, num_speakers, min_duration)
            preds[Path(wav).stem] = ann
        except Exception as e:
            print(f"⚠️ erro em {wav}: {e}")
    return preds

from pyannote.audio.utils.metric import DiscreteDiarizationErrorRate

def compute_der_with_protocol(protocol, predictions: dict, subset="test"):
    metric = DiscreteDiarizationErrorRate()
    used, skipped = 0, []
    for file in getattr(protocol, subset)():
        uri = file["uri"]
        if uri not in predictions:
            skipped.append(uri); continue
        ref = file["annotation"]   # ground-truth
        uem = file["annotated"]    # região válida
        hyp = predictions[uri]
        try:
            metric(ref, hyp, uem=uem); used += 1
        except Exception as e:
            skipped.append(uri)
    comps = metric.components()
    if comps["total"] == 0:
        print("❌ total=0 (nenhum arquivo válido)"); return None, used, skipped
    return float(abs(metric)), used, skipped

from pyannote.core import Annotation

def read_rttm_file(rttm_path):
    """
    Lê um arquivo RTTM e retorna {uri: Annotation}.
    """
    annotations = {}
    with open(rttm_path, "r") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.strip().split()
            if len(parts) < 9 or parts[0] != "SPEAKER":
                continue
            uri = parts[1]
            start = float(parts[3])
            duration = float(parts[4])
            speaker = parts[7]
            ann = annotations.setdefault(uri, Annotation(uri=uri))
            ann[Segment(start, start + duration)] = speaker
    return annotations

def load_reference_rttms(rttm_dir):
    """
    Lê todos os RTTMs de um diretório e retorna {uri: Annotation}.
    """
    refs = {}
    for path in Path(rttm_dir).glob("*.rttm"):
        anns = read_rttm_file(path)
        refs.update(anns)
    return refs

def compute_der_with_rttm_refs(refs: dict, preds: dict):
    metric = DiscreteDiarizationErrorRate()
    used, skipped = 0, []
    for uri, ref in refs.items():
        hyp = preds.get(uri)
        if hyp is None:
            skipped.append(uri); continue
        try:
            metric(ref, hyp)  # sem UEM, se não tiver
            used += 1
        except Exception:
            skipped.append(uri)
    comps = metric.components()
    if comps["total"] == 0:
        print("❌ total=0 (nenhum arquivo válido)"); return None, used, skipped
    return float(abs(metric)), used, skipped

### testes individuais

In [ ]:
from pathlib import Path
from pyannote.audio import Pipeline, Model
import torch

model_ft = os.environ.get("LOCAL_MODEL_CKPT", "lightning_logs/version_0/checkpoints/best.ckpt")
audio = os.environ.get("TEST_AUDIO_PATH", "dataset/audios/sample.wav")

seg_model = Model.from_pretrained(model_ft, strict=False)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

pipe_ft = Pipeline.from_pretrained("fatymatariq/speaker-diarization-3.1").to(device)

pipe_ft._segmentation.model = seg_model.to(device)

diarization = pipe_ft(
    audio,
    num_speakers=2
)

segmenta_por_speaker(
    audio,
    diarization
)
# with open("teste-ft-pytorch.rttm", "w") as f:
#     diarization.write_rttm(f)

In [ ]:
from pathlib import Path
from pyannote.audio import Pipeline, Model
import torch
from diarizers import SegmentationModel

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

seg_model = SegmentationModel().from_pretrained(os.environ["HF_FINETUNED_MODEL"], token=os.environ.get("HF_TOKEN", ""),).to(device)
seg_model = seg_model.to_pyannote_model().to(device)

pipe_ft = Pipeline.from_pretrained("fatymatariq/speaker-diarization-3.1").to(device)

pipe_ft._segmentation.model = seg_model.to(device)

diarization = pipe_ft(
    audio,
    num_speakers=2
)

segmenta_por_speaker(
    audio,
    diarization
)
# with open("teste-ft-diarizers.rttm", "w") as f:
#     diarization.write_rttm(f)

In [ ]:
from pathlib import Path
from pyannote.audio import Pipeline, Model
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

pipe_pre = Pipeline.from_pretrained("fatymatariq/speaker-diarization-3.1").to(device)

diarization_pre = pipe_pre(audio, num_speakers=2)

segmenta_por_speaker(
    audio,
    diarization_pre
)
# with open("teste-normal.rttm", "w") as f:
#     diarization_pre.write_rttm(f)

In [ ]:
from pathlib import Path
from pyannote.audio import Pipeline, Model
import torch
from diarizers import SegmentationModel
from datasets import load_dataset

audio = os.environ.get("TEST_AUDIO_PATH", "dataset/audios/sample.wav")

seg_model = SegmentationModel().from_pretrained(os.environ["HF_FINETUNED_MODEL"], token=os.environ.get("HF_TOKEN", ""),)
seg_model = seg_model.to_pyannote_model()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

pipe_ft = Pipeline.from_pretrained("fatymatariq/speaker-diarization-3.1").to(device)

pipe_ft._segmentation.model = seg_model.to(device)

dataset = load_dataset(os.environ["HF_DATASET_NAME"], split="test")

### teste grupo

In [ ]:
from pathlib import Path
from pyannote.audio import Pipeline, Model
import torch

# 1) Monte a lista de áudios a partir dos RTTMs de teste
test_names = {p.stem for p in Path("dataset/rttms/test").glob("*.rttm")}
AUDIO_LIST = [Path("dataset/audios") / f"{name}.wav" for name in test_names]
AUDIO_LIST = [p for p in AUDIO_LIST if p.exists()]  # filtra só os que existem
AUDIO_LIST = AUDIO_LIST[:50]

print(f"{len(AUDIO_LIST)} áudios encontrados para teste.")

In [ ]:
# 2) Dispositivo
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# 3) Pipelines (pretrained e finetuned)
pipe_pre = Pipeline.from_pretrained("fatymatariq/speaker-diarization-3.1").to(device)

preds_pre = run_model_over_list(pipe_pre, AUDIO_LIST, num_speakers=2, min_duration=1.0)

out_pre = Path("preds_pre"); out_pre.mkdir(parents=True, exist_ok=True)
for uri, ann in preds_pre.items():
    save_rttm(ann, out_pre / f"{uri}.rttm")

In [ ]:
pipe_ft = Pipeline.from_pretrained("fatymatariq/speaker-diarization-3.1").to(device)
seg_model = Model.from_pretrained(
    os.environ.get("LOCAL_MODEL_CKPT", "lightning_logs/version_0/checkpoints/best.ckpt")
).to(device)
# ponto correto de injeção do modelo de segmentação
pipe_ft._segmentation.model = seg_model

# 4) Rodar e salvar (supondo que você já tem as helpers abaixo definidas)
preds_ft  = run_model_over_list(pipe_ft,  AUDIO_LIST, num_speakers=2, min_duration=1.0)

out_ft  = Path("preds_ft");  out_ft.mkdir(parents=True, exist_ok=True)
for uri, ann in preds_ft.items():
    save_rttm(ann, out_ft / f"{uri}.rttm")


In [ ]:
from pathlib import Path
from typing import Dict
from pyannote.core import Annotation, Segment, Timeline
from pyannote.audio.utils.metric import DiscreteDiarizationErrorRate

# ---- leitores auxiliares ----
def read_rttm_file(rttm_path: Path) -> Dict[str, Annotation]:
    anns = {}
    with open(rttm_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 9 or parts[0] != "SPEAKER":
                continue
            uri   = parts[1]
            start = float(parts[3]); dur = float(parts[4])
            spk   = parts[7]
            ann = anns.setdefault(uri, Annotation(uri=uri))
            ann[Segment(start, start + dur)] = spk
    return anns

def load_rttm_dir(rttm_dir: Path) -> Dict[str, Annotation]:
    out = {}
    for p in sorted(Path(rttm_dir).glob("*.rttm")):
        out.update(read_rttm_file(p))
    return out

def read_uem_file(uem_path: Path) -> Dict[str, Timeline]:
    uems = {}
    with open(uem_path, "r") as f:
        for line in f:
            if not line.strip() or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 4:
                continue
            uri = parts[0]; start = float(parts[2]); end = float(parts[3])
            tl = uems.setdefault(uri, Timeline(uri=uri))
            tl.add(Segment(start, end))
    return uems

def load_uem_dir(uem_dir: Path) -> Dict[str, Timeline]:
    out = {}
    for p in sorted(Path(uem_dir).glob("*.uem")):
        out.update(read_uem_file(p))
    return out

# ---- "test" versão RTTM vs RTTM ----
def test_rttm(ref_rttm_dir: str | Path,
              pred_rttm_dir: str | Path,
              uem_dir: str | Path | None = None) -> float:
    """Compara diretórios de RTTMs previstos vs originais e retorna DER global."""
    refs  = load_rttm_dir(Path(ref_rttm_dir))
    preds = load_rttm_dir(Path(pred_rttm_dir))
    uems  = load_uem_dir(Path(uem_dir)) if uem_dir is not None else None

    metric = DiscreteDiarizationErrorRate()
    used, skipped = 0, []

    for uri, ref in refs.items():
        hyp = preds.get(uri)
        if hyp is None:
            skipped.append(uri); continue
        try:
            uem = uems.get(uri) if uems is not None else None
            metric(ref, hyp, uem=uem)
            used += 1
        except Exception:
            skipped.append(uri)

    if used == 0:
        raise RuntimeError("Nenhum par (ref, pred) válido encontrado.")

    der = float(abs(metric))
    print(f"DER={der:.4f} | usados={used} | pulados={len(skipped)}")
    return der
